# 003 · Readout amplitude sweep — solution

**Level:** medium · **Topics:** Gen2, SHFQA, readout, sweeps, chunking · **Type:** fix the bugs

Solution of [exercise.ipynb](exercise.ipynb).

The readout generator of the SHFQA / SHFQC can play only **one waveform per signal** in a real-time run, and the amplitude is part of the waveform. A real-time amplitude sweep needs ten different waveforms, so the compiler stops.

Two fixes are shown below: a near-time sweep, and a chunked real-time sweep.

In [ ]:
from laboneq.simple import *

## Device setup

One SHFQC+ with one qubit: a drive line and a readout line.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFQC(uid="shfqc", address="DEV12001", device_options="SHFQC/PLUS/QC2CH")
)
setup.add_connections(
    "shfqc",
    create_connection(to_signal="q0/drive", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html)

In [ ]:
calibration = Calibration()
calibration["q0/drive"] = SignalCalibration(
    oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=5.0e9),
    range=0,
)
calibration["q0/measure"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
calibration["q0/acquire"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6),
    local_oscillator=Oscillator(frequency=7.0e9),
    range=0,
)
setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Pulse and sweep

📖 [Pulse library](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/01_pulse_library.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html)

In [ ]:
readout_pulse = pulse_library.const(length=1e-6, amplitude=1.0)

amplitudes = LinearSweepParameter(start=0.1, stop=1.0, count=10)

## Fix A: near-time sweep

📖 [Measurement rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/03_measurement_rules.html) ·
[Averaging and sweeping](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/04_averaging_sweeping.html) ·
[Experiment chunking](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/tutorials/01_experiment_chunking.html)

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def readout_amplitude_sweep_near_time():
    # The sweep is outside the real-time loop: it runs in near-time
    with dsl.sweep(parameter=amplitudes) as amplitude:
        with dsl.acquire_loop_rt(count=16):
            with dsl.section(name="readout"):
                dsl.measure(
                    measure_signal="measure",
                    acquire_signal="acquire",
                    measure_pulse=readout_pulse,
                    measure_pulse_amplitude=amplitude,
                    integration_kernel=readout_pulse,
                    handle="result",
                    reset_delay=1e-6,
                )


experiment = readout_amplitude_sweep_near_time()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)
print(results["result"].axis)

**Expected result:** shape `(10,)` and an axis from 0.1 to 1.0.

## Fix B: chunked real-time sweep

In [ ]:
@dsl.experiment(signals=["drive", "measure", "acquire"])
def readout_amplitude_sweep_chunked():
    with dsl.acquire_loop_rt(count=16):
        # One chunk per sweep point: each real-time run has a single amplitude
        with dsl.sweep(parameter=amplitudes, chunk_count=10) as amplitude:
            with dsl.section(name="readout"):
                dsl.measure(
                    measure_signal="measure",
                    acquire_signal="acquire",
                    measure_pulse=readout_pulse,
                    measure_pulse_amplitude=amplitude,
                    integration_kernel=readout_pulse,
                    handle="result",
                    reset_delay=1e-6,
                )


experiment = readout_amplitude_sweep_chunked()
experiment.set_signal_map(
    {
        "drive": "q0/drive",
        "measure": "q0/measure",
        "acquire": "q0/acquire",
    }
)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

In [ ]:
# check
print(results["result"].data.shape)
print(results["result"].axis)

**Expected result:** shape `(10,)` and an axis from 0.1 to 1.0.

## Answers

1. On the drive line (SG channel) the sequencer has a command table: one waveform is stored and its amplitude is changed in real time.
   The readout generator has no command table. It stores the waveform of each readout signal and plays it as it is, so every amplitude is a new waveform, and only one is allowed per real-time run.
2. Move `dsl.sweep` outside `dsl.acquire_loop_rt`. Each amplitude is then a separate real-time run, and the waveform is uploaded again in between.
3. Only `chunk_count=10` works: one chunk per sweep point. With 2 or 5 chunks a single run still contains several amplitudes, and the same error comes back.
4. Very little here: in both cases the instrument does ten real-time runs with one amplitude each. Chunking keeps the experiment written as a real-time sweep, which helps when other real-time sweeps are nested inside. The price is the same for both: a waveform upload per point, so it is much slower than a true real-time sweep.

**Takeaway:** pulse parameters on the readout line (amplitude, length, phase) cannot change within one real-time run. Sweep them in near-time, or with one chunk per point.

📖 [Experiment chunking](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/tutorials/01_experiment_chunking.html) ·
[Measurement rules](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/03_measurement_rules.html)